# Projeto NEO — Previsão de Objetos Perigosos Próximos da Terra

Notebook único do projeto, cobrindo as fases iniciais do CRISP-DM:

1. **Business Understanding**
2. **Data Understanding**
3. **Data Preparation**

Dataset: `neo.csv` (NASA — Near-Earth Objects)

---
# 1. Business Understanding

**Tipo de tarefa:** Classificação binária supervisionada — prever se um objeto próximo da Terra (NEO) é potencialmente perigoso.

**Entidade das previsões:** Cada NEO (Near-Earth Object) individual — asteroide ou cometa identificado por `id`/`name`.

**Possíveis resultados a prever:** `hazardous = True` (potencialmente perigoso) ou `hazardous = False` (não perigoso).

**Quando são observados os resultados:** O rótulo `hazardous` já vem definido no dataset histórico, atribuído pela NASA/JPL com base em critérios orbitais (distância mínima de interseção orbital — MOID) e físicos (tamanho estimado do objeto). Não é o modelo que "calcula" o perigo em tempo real — o modelo aprende, a partir de exemplos já rotulados, um padrão que depois aplica a objetos novos ainda não classificados.

---
# 2. Data Understanding

Exploração de dados (EDA) do dataset `neo.csv`.

**Estrutura desta secção:**
1. Carregar os dados
2. Visão geral do dataset
3. Qualidade dos dados (valores em falta, duplicados)
4. Estatísticas descritivas
5. Colunas constantes / sem valor preditivo
6. Distribuição da variável-alvo
7. Distribuição das variáveis numéricas
8. Deteção de outliers
9. Comparação por classe
10. Matriz de correlação
11. Resumo dos insights

## 1. Carregar os dados

Se estiveres a correr no Google Colab, corre a célula seguinte para fazeres upload do ficheiro `neo.csv` a partir do teu computador. Se já tiveres o ficheiro no Google Drive, podes montar o Drive em alternativa.

In [ ]:
# Descomenta as linhas seguintes se precisares de fazer upload do ficheiro no Colab
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

df = pd.read_csv("neo.csv")
df.head()

## 2. Visão geral do dataset

In [ ]:
print("Dimensões (linhas, colunas):", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nInformação geral:")
df.info()

## 3. Qualidade dos dados

### 3.1 Valores em falta

In [ ]:
print("Valores em falta por coluna:")
print(df.isnull().sum())

print("\nPercentagem de valores em falta:")
print((df.isnull().sum() / len(df) * 100).round(2))

### 3.2 Registos duplicados

In [ ]:
print("Linhas totalmente duplicadas:", df.duplicated().sum())
print("IDs duplicados:", df['id'].duplicated().sum())

## 4. Estatísticas descritivas (variáveis numéricas)

In [ ]:
df.describe().T

## 5. Colunas constantes / sem valor preditivo

Colunas como `orbiting_body` e `sentry_object` podem não variar no dataset, o que as torna irrelevantes para o modelo.

In [ ]:
for col in ['orbiting_body', 'sentry_object']:
    print(f"{col}: valores únicos = {df[col].unique()}")

## 6. Distribuição da variável-alvo (`hazardous`)

Esta é a variável que o modelo vai prever. É importante perceber desde já se as classes estão equilibradas, pois isso condiciona as métricas de avaliação a usar mais tarde (ex: accuracy pode ser enganadora em datasets desequilibrados).

In [ ]:
print(df['hazardous'].value_counts())
print("\nPercentagem:")
print((df['hazardous'].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(5,4))
sns.countplot(data=df, x='hazardous')
plt.title("Distribuição da variável-alvo (hazardous)")
plt.xlabel("Perigoso")
plt.ylabel("Contagem")
plt.show()

## 7. Distribuição das variáveis numéricas

In [ ]:
num_cols = ['est_diameter_min', 'est_diameter_max', 'relative_velocity',
            'miss_distance', 'absolute_magnitude']

df[num_cols].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()

## 8. Deteção de outliers (boxplots)

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 9. Comparação das variáveis por classe (`hazardous`)

Permite perceber visualmente se alguma variável separa bem as duas classes.

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x='hazardous', y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 10. Matriz de correlação

In [ ]:
plt.figure(figsize=(8, 6))
corr_df = df[num_cols + ['hazardous']].copy()
corr_df['hazardous'] = corr_df['hazardous'].astype(int)
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlação")
plt.show()

## 11. Resumo dos insights

In [ ]:
print("RESUMO DO DATA UNDERSTANDING")
print("-" * 40)
print(f"Total de registos: {len(df)}")
print(f"Total de colunas: {df.shape[1]}")
print(f"Valores em falta: {df.isnull().sum().sum()}")
print(f"Linhas duplicadas: {df.duplicated().sum()}")
print(f"Proporção de perigosos (True): {df['hazardous'].mean()*100:.2f}%")
print(f"orbiting_body é constante? {df['orbiting_body'].nunique() == 1}")
print(f"sentry_object é constante? {df['sentry_object'].nunique() == 1}")

## Conclusões (a preencher)

- O dataset tem **90.836 registos** e **10 colunas**, sem valores em falta.
- A variável-alvo `hazardous` está **desequilibrada** (~90% não perigosos vs. ~10% perigosos) — a considerar na fase de modelação.
- As colunas `orbiting_body` e `sentry_object` são **constantes** neste dataset e não têm valor preditivo — candidatas a remoção na fase de *Data Preparation*.
- As colunas `id` e `name` são identificadores e não devem ser usadas como *features*.
- _(completar com as observações tiradas dos gráficos: quais variáveis parecem separar melhor as classes, se há correlações fortes entre `est_diameter_min` e `est_diameter_max`, outliers relevantes, etc.)_

---
# 3. Data Preparation

Preparação do dataset com base nos insights da secção anterior.

**Insights relevantes do Data Understanding:**
- Sem valores em falta nem duplicados.
- `orbiting_body` e `sentry_object` são colunas constantes (sem valor preditivo).
- `id` e `name` são identificadores (sem valor preditivo).
- Classe-alvo `hazardous` desequilibrada: ~90% False vs. ~10% True.

**Estrutura desta secção:**
1. Carregar os dados
2. Remover colunas irrelevantes
3. Verificar colunas redundantes (correlação)
4. Converter a variável-alvo
5. Separar em treino e teste (antes de qualquer transformação, para evitar *data leakage*)
6. Tratar outliers
7. Escalar as variáveis numéricas
8. Lidar com o desequilíbrio de classes
9. Guardar os datasets preparados

In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("neo.csv")
df.shape

## 2. Remover colunas irrelevantes

- `id`, `name`: identificadores, não têm poder preditivo.
- `orbiting_body`, `sentry_object`: constantes no dataset (o Data Understanding confirmou que só têm um valor).

In [ ]:
cols_to_drop = ['id', 'name', 'orbiting_body', 'sentry_object']
df = df.drop(columns=cols_to_drop)
df.head()

## 3. Verificar colunas redundantes

`est_diameter_min` e `est_diameter_max` são derivadas diretamente da mesma medida (magnitude absoluta) e estão fortemente correlacionadas. Vamos confirmar e decidir se removemos uma delas para evitar redundância/multicolinearidade.

In [ ]:
print(df[['est_diameter_min', 'est_diameter_max']].corr())

A correlação é (praticamente) perfeita. Vamos manter apenas uma delas — `est_diameter_max` — e criar uma nova feature `diameter_mean` como alternativa a testar mais tarde na fase de Modeling.

In [ ]:
df['diameter_mean'] = (df['est_diameter_min'] + df['est_diameter_max']) / 2
df = df.drop(columns=['est_diameter_min'])
df.head()

## 4. Converter a variável-alvo

`hazardous` está como booleano (`True`/`False`). Convertemos para `1`/`0`, formato que a generalidade dos algoritmos de classificação espera.

In [ ]:
df['hazardous'] = df['hazardous'].astype(int)
df['hazardous'].value_counts()

## 5. Separar em treino e teste

Fazemos o *split* **antes** de escalar ou tratar outliers, para que essas transformações sejam "aprendidas" apenas com os dados de treino e depois aplicadas ao teste — evita *data leakage*.

Usamos `stratify=y` porque a classe-alvo está desequilibrada, garantindo que treino e teste mantêm a mesma proporção de casos perigosos/não perigosos.

In [ ]:
X = df.drop(columns=['hazardous'])
y = df['hazardous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("\nProporção de hazardous no treino:")
print(y_train.value_counts(normalize=True))
print("\nProporção de hazardous no teste:")
print(y_test.value_counts(normalize=True))

## 6. Tratar outliers

No Data Understanding vimos que `est_diameter_max` e `relative_velocity` têm outliers (~9% e ~2%, respetivamente), mas `miss_distance` e `absolute_magnitude` praticamente não têm.

Como os outliers em `est_diameter_max` podem corresponder a objetos genuinamente grandes (fisicamente plausíveis, não erros de medição), **não os vamos remover** — remover instâncias reais de objetos grandes iria enviesar o modelo, e são precisamente os objetos maiores que mais interessa detetar como potencialmente perigosos. Em vez de remover, usamos escalonamento robusto a outliers na secção seguinte.

> Nota: os limites do IQR são calculados apenas com o treino, para não "espreitar" o teste.

In [ ]:
num_cols = ['diameter_mean', 'est_diameter_max', 'relative_velocity',
            'miss_distance', 'absolute_magnitude']

for col in num_cols:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((X_train[col] < lo) | (X_train[col] > hi)).sum()
    print(f"{col}: {n_out} outliers ({n_out/len(X_train)*100:.1f}%) | limites: [{lo:.2f}, {hi:.2f}]")

## 7. Escalar as variáveis numéricas

Usamos `StandardScaler`, ajustado (`fit`) apenas no treino e depois aplicado (`transform`) a treino e teste.

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled.describe().T

## 8. Lidar com o desequilíbrio de classes

A classe `hazardous=1` representa apenas ~10% dos dados. Isto será tratado principalmente na fase de **Modeling** (ex: `class_weight='balanced'`, SMOTE, undersampling), mas fica aqui documentado como decisão de preparação a levar em conta.

Se quiseres já aplicar SMOTE nesta fase (apenas no treino, nunca no teste), a biblioteca `imbalanced-learn` disponibiliza isso:

```python
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
```

Fica comentado por agora — discutam em grupo se preferem tratar isto aqui ou apenas na fase de Modeling (com `class_weight`, por exemplo).

## 9. Guardar os datasets preparados

In [ ]:
X_train_scaled.to_csv('X_train.csv', index=False)
X_test_scaled.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Ficheiros guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv")

# No Colab, se quiseres descarregar os ficheiros:
# from google.colab import files
# files.download('X_train.csv')
# files.download('X_test.csv')
# files.download('y_train.csv')
# files.download('y_test.csv')

## Resumo das decisões tomadas

- Removidas colunas `id`, `name`, `orbiting_body`, `sentry_object` (sem valor preditivo).
- Removida `est_diameter_min` por redundância quase perfeita com `est_diameter_max`; criada `diameter_mean` como feature alternativa.
- `hazardous` convertida para inteiro (0/1).
- Split treino/teste (80/20) feito **antes** de qualquer transformação, com `stratify` para preservar a proporção de classes.
- Outliers não removidos (correspondem a objetos fisicamente plausíveis, importantes para o problema).
- Variáveis numéricas escalonadas com `StandardScaler` (ajustado só no treino).
- Desequilíbrio de classes documentado, a tratar na fase de Modeling (ou com SMOTE já aqui, se o grupo preferir).